# Dhara — act-title documents + cross-encoder reranking

Two changes, neither of which needs any training, run in one pass because the
second consumes the first's output.

The zero-shot dense control sits at provision-level R@10 = 0.324 over 552
human-adjudicated questions and all 39,484 chunks. Error analysis of that run
says the losses split three ways:

| bucket | count | share |
|---|---|---|
| already in top 10 | 179 | 32.4% |
| **ranks 11-100 — a reranker's job** | **167** | **30.3%** |
| ranks 101-200 | 38 | 6.9% |
| never retrieved (beyond 200) | 168 | 30.4% |

### Stage 1 — put the Act title in the document

The zero-shot index embedded `provision_title_bn + text_bn` and nothing else,
so the encoder never saw which statute a section belongs to. A citizen question
about divorce has to reach *The Muslim Family Laws Ordinance, 1961*, and the
words "Muslim Family Laws Ordinance" were not in the embedded text at all. This
matters most exactly where the system is weakest: 40.2% of English-gold
questions currently have their answer beyond rank 200, and 60% of all gold
answers live in English-only Acts whose *titles* are often the strongest
available bridge from a Bangla question.

Cheap to test: one re-embed, no training, and it is a clean ablation against
the existing control because only the document template changes.

### Stage 2 — cross-encoder rerank

A bi-encoder compares two vectors computed independently. A cross-encoder reads
the question and the provision *together*, which is far more accurate and far
too slow to run over 39,484 chunks — so it reranks the top 100 the bi-encoder
already found. That is the standard second stage and this project has never
run it.

`BAAI/bge-reranker-v2-m3` is the matching reranker for BGE-m3: same
multilingual training lineage, so it should carry the same cross-language
ability that made BGE-m3 the right retriever in the first place. Zero-shot, no
fine-tuning.

**The ceiling is real and gets reported.** Reranking the top 100 cannot fix the
30.4% of questions whose gold provision was never retrieved. Even a perfect
reranker caps R@10 at 0.627 here. `21_eval_rerank.py` records that bound next
to the achieved number so the two are never confused.

### Upload

`corpus_v1.jsonl.gz` and `probe_questions.jsonl`. That is all — no training
data, because nothing here trains.

In [1]:
!pip -q install "sentence-transformers>=3.0"

In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gzip, json, time, collections
import numpy as np, torch

REQUIRED = ["probe_questions.jsonl"]
def have_corpus(): return os.path.exists("corpus_v1.jsonl") or os.path.exists("corpus_v1.jsonl.gz")

if not have_corpus() or any(not os.path.exists(f) for f in REQUIRED):
    try:
        from google.colab import files; files.upload()
    except Exception:
        pass

assert have_corpus(), "missing corpus_v1.jsonl(.gz)"
for f in REQUIRED:
    assert os.path.exists(f), f"missing {f}"

def read_jsonl(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt", encoding="utf-8") as fh:
        for ln, line in enumerate(fh, 1):
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except Exception as e:
                raise SystemExit(f"{path} line {ln} corrupt ({e}) - re-upload the .gz")
    return rows

CORPUS_PATH = "corpus_v1.jsonl.gz" if os.path.exists("corpus_v1.jsonl.gz") else "corpus_v1.jsonl"
assert torch.cuda.is_available(), "no GPU - Runtime > Change runtime type > T4 GPU"
DEV = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

chunks = read_jsonl(CORPUS_PATH)
probes = read_jsonl("probe_questions.jsonl")
assert len(chunks) == 39484, f"expected 39484 chunks, got {len(chunks)}"
assert len(probes) == 552,   f"expected 552 probe questions, got {len(probes)}"
cid   = [c["chunk_id"] for c in chunks]
cprov = [c["provision_id"] for c in chunks]
print(f"{len(chunks)} chunks | {len(probes)} probe questions")

Saving corpus_v1.jsonl.gz to corpus_v1.jsonl.gz
Saving probe_questions.jsonl to probe_questions.jsonl
GPU: Tesla T4
39484 chunks | 552 probe questions


## Stage 1: documents with the Act title

`act_title_bn` for Bangla statutes, `act_title_en` for the English-only ones,
and both when both exist — a Bangla question may need either as its bridge, and
BGE-m3 is multilingual precisely so it can hold the two together.

The old template is kept alongside so the difference is visible in this
notebook, not just inferred from two separate runs.

In [3]:
def document_old(c):
    """The zero-shot control's template. Kept for comparison only."""
    return f"{(c.get('provision_title_bn') or '')} {c['text_bn']}".strip()

def document_new(c):
    """Act title(s) + provision title + body."""
    bits = []
    for key in ("act_title_bn", "act_title_en"):
        v = (c.get(key) or "").strip()
        if v and v not in bits:
            bits.append(v)
    t = (c.get("provision_title_bn") or "").strip()
    if t:
        bits.append(t)
    bits.append(c["text_bn"])
    return " ".join(b for b in bits if b).strip()

text_new = {c["chunk_id"]: document_new(c) for c in chunks}

for c in chunks[:2]:
    print("OLD:", document_old(c)[:150].replace("\n", " "))
    print("NEW:", document_new(c)[:150].replace("\n", " "))
    print()

lens = [len(v) for v in text_new.values()]
print(f"new document chars: median {int(np.median(lens))}, mean {int(np.mean(lens))}")

OLD: Estates of Hindus, Muhammadans and others, not being disqualified landholders, leaving wills In all cases of Hindu, Mussalman or other person subject 
NEW: THE1[***] WILLS AND INTESTACY REGULATION, 1799 THE [***] WILLS AND INTESTACY REGULATION, 1799 Estates of Hindus, Muhammadans and others, not being dis

OLD: Estates of persons dying intestate In case of a Hindu, Mussalman or other person subject to the jurisdiction of the Zila Courts dying intestate, but l
NEW: THE1[***] WILLS AND INTESTACY REGULATION, 1799 THE [***] WILLS AND INTESTACY REGULATION, 1799 Estates of persons dying intestate In case of a Hindu, M

new document chars: median 550, mean 757


In [4]:
from sentence_transformers import SentenceTransformer

CHECKPOINT = "BAAI/bge-m3"
MAX_SEQ    = 512      # identical to the zero-shot control

embedder = SentenceTransformer(CHECKPOINT, device=DEV)
embedder.max_seq_length = MAX_SEQ

t = time.time()
C = embedder.encode([text_new[c] for c in cid], batch_size=128,
                    normalize_embeddings=True, convert_to_numpy=True,
                    show_progress_bar=True).astype(np.float16)
print(f"corpus embedded {C.shape} in {(time.time()-t)/60:.1f} min")

Q = embedder.encode([p["question_bn"] for p in probes], batch_size=64,
                    normalize_embeddings=True, convert_to_numpy=True).astype(np.float16)
print("queries embedded", Q.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/309 [00:00<?, ?it/s]

corpus embedded (39484, 1024) in 38.3 min
queries embedded (552, 1024)


In [5]:
# Retrieve deep once: top-200 feeds both the stage-1 score and the rerank pool.
DEPTH = 200
Cg = torch.tensor(C, device=DEV, dtype=torch.float16)
Qg = torch.tensor(Q, device=DEV, dtype=torch.float16)
dense_rank = []
for i in range(0, len(probes), 64):
    s = Qg[i:i+64] @ Cg.T
    dense_rank.append(torch.topk(s, DEPTH, dim=1).indices.cpu().numpy())
dense_rank = np.vstack(dense_rank)
del Cg, Qg; torch.cuda.empty_cache()
print("dense_rank", dense_rank.shape)

KS = (1, 5, 10, 20, 50, 100)
def score(get_order):
    agg = {k: collections.Counter() for k in KS}; tot = collections.Counter()
    for i, p in enumerate(probes):
        tag = p["lang_tag"]; tot[tag] += 1; tot["all"] += 1
        gold = set(p["gold_provision_ids"])
        rank = next((r for r, ci in enumerate(get_order(i), 1) if cprov[ci] in gold), None)
        for k in KS:
            if rank and rank <= k:
                agg[k][tag] += 1; agg[k]["all"] += 1
    return {g: {f"R@{k}": round(agg[k][g]/tot[g], 3) for k in KS}
            for g in ("all", "english", "bengali", "mixed")}

acttitle = score(lambda i: dense_rank[i])
print("\n=== STAGE 1: dense with Act title ===")
for g, row in acttitle.items():
    print(f"  {g:8s} " + "  ".join(f"{k}={v:.3f}" for k, v in row.items()))
print("\n=== control: dense WITHOUT Act title (locked 2026-08-31) ===")
print("  all      R@1=0.096  R@5=0.239  R@10=0.324  R@20=0.413  R@50=0.534  R@100=0.627")
print("  english  R@1=0.054  R@5=0.159  R@10=0.222  R@20=0.288  R@50=0.429  R@100=0.532")
print("  bengali  R@1=0.213  R@5=0.426  R@10=0.546  R@20=0.685  R@50=0.741  R@100=0.806")
print("  mixed    R@1=0.108  R@5=0.297  R@10=0.414  R@20=0.523  R@50=0.649  R@100=0.739")

dense_rank (552, 200)

=== STAGE 1: dense with Act title ===
  all      R@1=0.130  R@5=0.270  R@10=0.344  R@20=0.413  R@50=0.538  R@100=0.607
  english  R@1=0.084  R@5=0.186  R@10=0.231  R@20=0.279  R@50=0.408  R@100=0.498
  bengali  R@1=0.259  R@5=0.472  R@10=0.574  R@20=0.676  R@50=0.778  R@100=0.796
  mixed    R@1=0.144  R@5=0.324  R@10=0.459  R@20=0.559  R@50=0.694  R@100=0.748

=== control: dense WITHOUT Act title (locked 2026-08-31) ===
  all      R@1=0.096  R@5=0.239  R@10=0.324  R@20=0.413  R@50=0.534  R@100=0.627
  english  R@1=0.054  R@5=0.159  R@10=0.222  R@20=0.288  R@50=0.429  R@100=0.532
  bengali  R@1=0.213  R@5=0.426  R@10=0.546  R@20=0.685  R@50=0.741  R@100=0.806
  mixed    R@1=0.108  R@5=0.297  R@10=0.414  R@20=0.523  R@50=0.649  R@100=0.739


## Stage 2: cross-encoder rerank of the top 100

The reranker reads (question, provision) as one input, so it can weigh the
question's actual words against the statute's — which is exactly what a
bi-encoder's two independent vectors cannot do.

Rerank depth 100, not 200: the marginal candidates between 100 and 200 hold
only 6.9% of gold answers, and depth doubles the cross-encoder's cost, which
is the expensive part of this pipeline. Depth is recorded in the output so the
choice is visible rather than assumed.

In [6]:
from sentence_transformers import CrossEncoder

RERANKER = "BAAI/bge-reranker-v2-m3"
RERANK_DEPTH = 100
RERANK_BATCH = 32   # 568M cross-encoder at seq-512 on a 15GB T4. If this OOMs,
                    # drop to 16 -- unlike the bi-encoder's training batch, this
                    # is pure inference, so batch size changes speed only, never
                    # the result.

reranker = CrossEncoder(RERANKER, max_length=512, device=DEV)

t = time.time()
rerank_rows = []
for i, p in enumerate(probes):
    cand_idx = dense_rank[i][:RERANK_DEPTH]
    pairs = [(p["question_bn"], text_new[cid[j]]) for j in cand_idx]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    rerank_rows.append({
        "qid": p["qid"],
        "ranked_chunk_ids": [cid[cand_idx[o]] for o in order],
        "scores": [float(scores[o]) for o in order],
        "rank_before": int(next((r for r, j in enumerate(cand_idx, 1)
                                 if cprov[j] in set(p["gold_provision_ids"])), 0)) or None,
        "candidate_source": "bge_m3_acttitle_dense_top100",
        "reranker_checkpoint": RERANKER,
        "reranker_fine_tuned": False,
    })
    if (i + 1) % 100 == 0:
        rate = (time.time() - t) / (i + 1)
        print(f"  reranked {i+1}/{len(probes)}  ({(time.time()-t)/60:.1f} min elapsed, "
              f"~{rate*(len(probes)-i-1)/60:.1f} min left)")
print(f"reranked all {len(probes)} in {(time.time()-t)/60:.1f} min")

prov_of = {c: cprov[k] for k, c in enumerate(cid)}
pos = {c: k for k, c in enumerate(cid)}   # chunk_id -> row, so scoring stays O(1) per lookup
reranked = score(lambda i: [pos[c] for c in rerank_rows[i]["ranked_chunk_ids"]])
print("\n=== STAGE 2: after cross-encoder rerank ===")
for g, row in reranked.items():
    print(f"  {g:8s} " + "  ".join(f"{k}={v:.3f}" for k, v in row.items()))

reachable = sum(1 for i, p in enumerate(probes)
                if set(p["gold_provision_ids"]) & {prov_of[c] for c in rerank_rows[i]["ranked_chunk_ids"]})
print(f"\nceiling: gold present in the reranked candidates for {reachable}/{len(probes)} "
      f"= {reachable/len(probes):.3f} (reranking cannot exceed this)")
print("Below that ceiling, a miss is the reranker's. Above it, retrieval had")
print("already lost the answer before the reranker saw anything.")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

  reranked 100/552  (14.8 min elapsed, ~67.0 min left)
  reranked 200/552  (30.0 min elapsed, ~52.7 min left)
  reranked 300/552  (45.0 min elapsed, ~37.8 min left)
  reranked 400/552  (59.6 min elapsed, ~22.7 min left)
  reranked 500/552  (74.5 min elapsed, ~7.8 min left)
reranked all 552 in 82.4 min

=== STAGE 2: after cross-encoder rerank ===
  all      R@1=0.132  R@5=0.272  R@10=0.342  R@20=0.418  R@50=0.540  R@100=0.607
  english  R@1=0.093  R@5=0.174  R@10=0.219  R@20=0.273  R@50=0.402  R@100=0.498
  bengali  R@1=0.222  R@5=0.481  R@10=0.574  R@20=0.639  R@50=0.769  R@100=0.796
  mixed    R@1=0.162  R@5=0.360  R@10=0.486  R@20=0.640  R@50=0.730  R@100=0.748

ceiling: gold present in the reranked candidates for 335/552 = 0.607 (reranking cannot exceed this)
Below that ceiling, a miss is the reranker's. Above it, retrieval had
already lost the answer before the reranker saw anything.


In [7]:
# Save both stages. The index goes back in scripts/13's layout; the rerank goes
# in scripts/21's layout. Both are scored in-repo by the same code that scored
# every earlier rung, so the paired bootstrap can compare them directly.
np.save("embeddings.npy", C)
json.dump(cid, open("chunk_ids.json", "w"))
np.save("probe_query.npy", Q)
json.dump({
    "checkpoint": CHECKPOINT, "max_seq_length": MAX_SEQ,
    "use_title": True, "use_act_title": True,
    "query_prefix": "", "passage_prefix": "",
    "dim": int(C.shape[1]), "dtype": "float16", "normalized": True,
    "n_chunks": int(C.shape[0]), "corpus_file": "data/processed/corpus_v1.jsonl",
    "fine_tuned": False,
    "document_template": "act_title_bn + act_title_en + provision_title_bn + text_bn",
    "built_by": "notebooks/colab_acttitle_and_rerank.ipynb",
    "notes": "Act-title ablation against models/index_bge_m3_zeroshot_v1, whose template "
             "was provision_title_bn + text_bn only.",
}, open("manifest.json", "w"), indent=2)

with open("rerank_bge_v2m3_top100.jsonl", "w", encoding="utf-8") as fh:
    for r in rerank_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

try:
    from google.colab import files
    for f in ["embeddings.npy", "chunk_ids.json", "probe_query.npy",
              "manifest.json", "rerank_bge_v2m3_top100.jsonl"]:
        files.download(f)
except Exception:
    print("not on Colab - grab these from the file panel:")
    print("  embeddings.npy chunk_ids.json probe_query.npy manifest.json rerank_bge_v2m3_top100.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Back in the repo

```bash
# Stage 1 — the act-title index
mkdir -p models/index_bge_m3_acttitle_v1
mv embeddings.npy chunk_ids.json probe_query.npy manifest.json models/index_bge_m3_acttitle_v1/

python scripts/13_eval_dense_index.py \
    --index models/index_bge_m3_acttitle_v1 --run-id bge_m3_acttitle
python scripts/18_compare_runs.py \
    --a results/runs/bge_m3_acttitle.json \
    --b results/runs/bge_m3_zeroshot.json

# Stage 2 — the rerank
mv rerank_bge_v2m3_top100.jsonl data/processed/

python scripts/21_eval_rerank.py \
    --rerank data/processed/rerank_bge_v2m3_top100.jsonl \
    --run-id bge_m3_acttitle_rerank
python scripts/18_compare_runs.py \
    --a results/runs/bge_m3_acttitle_rerank.json \
    --b results/runs/bge_m3_zeroshot.json
```

Two comparisons, both against the same locked control, both with a paired
bootstrap CI on the difference. The second one is the number that matters for
the system: it is the full retrieve-then-rerank pipeline against the
retrieve-only baseline.

Do not read the rerank result without its `recoverable_ceiling`. Reranking
cannot retrieve what retrieval missed, and on this candidate set roughly 3
questions in 10 were already gone before the reranker saw anything.